In [0]:
%sql
SELECT viewing_type, COUNT(*), COUNT(DISTINCT ad_id) FROM (
SELECT *, DENSE_RANK() OVER (PARTITION BY viewing_type ORDER BY impression_count DESC) AS rk
FROM dev.mohit_gangwani.ad_viewing_type_dma_overall)
WHERE rk <= 10000
GROUP BY 1

In [0]:
%sql
WITH ovrl AS (
  SELECT ad_id, SUM(impression_count) AS ttl_count
  FROM dev.mohit_gangwani.ad_viewing_type_dma_overall
  WHERE viewing_type != 'Unknown'
  GROUP BY 1
)
, ad_filter AS (
  SELECT ad_id, ttl_count, DENSE_RANK() OVER (ORDER BY ttl_count DESC) AS rk
  FROM ovrl
)
SELECT a.*
FROM dev.mohit_gangwani.ad_viewing_type_dma_overall a
JOIN ad_filter f
  ON f.ad_id = a.ad_id
WHERE f.rk <= 10000
GROUP BY ALL

In [0]:
%sql
SELECT viewing_type, COUNT(*), COUNT(DISTINCT ad_id) FROM dev.mohit_gangwani.ad_viewing_type_dma_overall GROUP BY viewing_type 

In [0]:
%sql
SELECT * FROM prod.detection.dma
WHERE dma_id IN (220, 286)

In [0]:
%sql
SELECT DATE_TRUNC('HOUR', session_start), fk_commercial_source_id, COUNT(*)
FROM prod.detection.viewing_commercials_firehose_dedup vc
WHERE session_start >= CURRENT_DATE - 5
GROUP BY 1, 2

In [0]:
%sql
SELECT DATE_TRUNC('DAY', session_start), client_name, COUNT(*)
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.fk_commercial_id = vc.fk_commercial_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
WHERE session_start >= CURRENT_DATE - 5
GROUP BY 1, 2

In [0]:
%sql
SELECT DATE_TRUNC('DAY', session_start), commercial_client, COUNT(*)
FROM prod.detection.viewing_commercials_golden vc
WHERE session_start >= CURRENT_DATE - 5
AND session_start_hour >= CURRENT_DATE - 5
GROUP BY 1, 2